In [5]:
#pip install imap_tools

# В настройках Яндекс.Почты перейдите в «Все настройки» → «Почтовые программы» и убедитесь, что включен доступ по IMAP
# Перейдите на → «Пароли приложений» — создайте специальный пароль для своего скрипта Python.
# Используйте этот пароль в скрипте (не основной от Яндекса).

In [6]:
from imap_tools import MailBox
import pandas as pd
from io import BytesIO
from datetime import datetime, timedelta
import json
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import os

Настройка АПИ google

In [369]:
# подгружаем полученные credentials
path_to_credential = r'C:\Users\1\Desktop\Для работы с гугл таблицами API\YOUR_CRED.json' 

# Вводим название таблицы на гугле
table_name = 'Привлечение и Сторонний КЦ'

#тех данные API гугл sheets и drive
scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

credentials = ServiceAccountCredentials.from_json_keyfile_name(path_to_credential, scope)

#авторизация на gsheets
gs = gspread.authorize(credentials)

#назначаем переменную для открытия таблицы
work_sheet = gs.open(table_name)

# назначаем вкладке имя переменной
REPORT_CS = work_sheet.worksheet("Отчёт КЦ ТЕСТ")

ATTENTION
- РАБОТАЕТ С ПОСЛЕДНИМИ ДАТАМИ В ЛЮБОМ ОТЧЁТЕ ОТ КЦ - ХОТЬ НАКОПИТЕЛЬНЫЙ ХОТЬ НЕТ
НО! ТРЕБУЕТСЯ, ЧТОБЫ НОВЫХ ДАННЫХ ПО РАНЕЕ ПРИСЛАННЫМ ДАТАМ НЕ БЫЛО.
- ЕСЛИ ЗАВТРА БУДУТ НОВЫЕ СТРОКИ С ДАТОЙ, КОТОРУЮ ДОБАВЛЯЛИ СЕГОДНЯ ТО ИХ НЕ ПРИМЕТ
- ЕСЛИ СКРИПТ НЕ БУДЕТ ЗАПУЩЕН ПОСЛЕ ПРИХОДА НОВОЙ ВЫГРУЗКИ, А БУДЕТ ЗАПУЩЕН ПОСЛЕ ПРИХОДА СЛЕДУЮЩЕЙ ВЫГРУЗКИ, ТО БУДУТ СЧИТАТЬСЯ ТОЛЬКО СТРОКИ ПОСЛЕДНЕЙ ВЫГРУЗКИ.

In [8]:
# загружаем файл с последними подгруженными датами по коллцентрам
last_dates = pd.read_excel(r'C:\Users\1\Последние_даты_отчёт_КЦ.xlsx')

КЦ №1

In [178]:
# получаем данные

In [9]:
folder_name = 'КЦ_1'  # Имя папки
login = 'login@yandex.ru'
password = '******' # пароль для своего скрипта Python

with MailBox('imap.yandex.com').login(login, password, folder_name) as mailbox:
    # Получаем самое свежее письмо с вложением .xlsx в папке "КЦ"
    msg = next(mailbox.fetch(reverse=True, limit=1, mark_seen=False,
                             criteria='ALL'))
    for att in msg.attachments:
        if att.filename.endswith('.xlsx'):
            # Преобразуем вложение в DataFrame без сохранения на диск
            reply_cs = pd.read_excel(BytesIO(att.payload))

C:\Users\1\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [37]:
#убираем ненужные колонки
reply_cs = reply_cs[['Номер базы', 'Название базы', 'ID клиента', 'Название клиента', 'Регион', 'CallID', 'Телефон', 'Дата и время звонка', 'Результат звонка', 'Причина отказа', 'Комментарий', 'Расшифровка нецелевой', 'EMAIL со слов клиента', 'Время общения с оператором, мин', 'Статус']]

In [39]:
# обозначаем переменную равную последнему дню, заргуженному у реплая
reply_last_date = last_dates.loc[last_dates['KC'] == 'replay', 'last_date'].iloc[0].date()

In [41]:
#выбираем только даты, которые больше последней даты по реплаю
reply_cs = reply_cs[reply_cs['Дата и время звонка'].dt.date > reply_last_date]

In [47]:
#Проверка на 10 знаков в телефоне

In [45]:
if reply_cs['ID клиента'].count() == 0:
    print('Незагруженные ранее данные отсутствуют!')
else:
    #вставляем в датафрейм с последними датами ту дату, которой ещё не было и которую мы загружаем в гугл документ
    last_dates.loc[last_dates['KC'] == 'replay', 'last_date'] = reply_cs['Дата и время звонка'].max().strftime("%d.%m.%Y")
    print(f'{reply_cs['ID клиента'].count()} строк готовы к загрузке!')

Незагруженные ранее данные отсутствуют!


In [357]:
# убираем первую цифру от телефона
reply_cs['Телефон'] = reply_cs['Телефон'].astype(str).str[1:]

In [515]:
# Получение всех значений в нужном столбце, например, в колонке A
column_values = REPORT_CS.col_values(1)  # Нумерация столбцов с 1 (A=1)

# Поиск первой пустой строки
first_empty_row = len(column_values) + 1  # следующая строка после последней непустой

# Переменная вставки с этой строки и колонки A (1)
start_cell = f'A{first_empty_row}'

In [541]:
# загрузка в гугл

for col in reply_cs.select_dtypes(include=['datetime', 'timedelta']).columns:
    reply_cs[col] = reply_cs[col].dt.strftime("%d.%m.%Y %H:%M:%S")
        # Заменяем NaN значениями пустыми строками
reply_cs.fillna('', inplace=True)  

REPORT_CS.update(start_cell, reply_cs.values.tolist())
print(f'Данные загружены отчёт КЦ с {start_cell} ячейки {pd.to_datetime("today")}')


C:\Users\1\AppData\Local\Temp\ipykernel_15500\3531549687.py:8: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  REPORT_CS.update(start_cell, reply_cs.values.tolist())


Данные загружены с A234 ячейки 2025-11-21 17:24:57.002971


КЦ №2

In [ ]:
# получаем данные

In [11]:
folder_name_2 = 'КЦ_2'  # Имя  папки

with MailBox('imap.yandex.com').login(login, password, folder_name_2) as mailbox:
    # Получаем самое свежее письмо с вложением .xlsx в папке "КЦ"
    msg_2 = next(mailbox.fetch(reverse=True, limit=1, mark_seen=False,
                             criteria='ALL'))
    for att in msg_2.attachments:
        if att.filename.endswith('.xlsx'):
            # Преобразуем вложение в DataFrame без сохранения на диск
            oline_cs = pd.read_excel(BytesIO(att.payload))

In [13]:
oline_cs = oline_cs[['номер базы', 'название базы', 'id клиента', 'название компании', 'регион', 'Ссылка на звонок', 'Номер телефона', 'Дата и время', 'Результат звонка', 'Расшифровка', 'Комментарий','EMAIL со слов клиента', 'Время разговора в поминутной тарификации', 'Статус звонка']]

In [15]:
# Создать новую колонку с пустыми значениями для Олайн
oline_cs['Расшифровка нецелевой'] = ''

# определим порядок колонок
cols = ['номер базы', 'название базы', 'id клиента', 'название компании', 'регион',
        'Ссылка на звонок', 'Номер телефона', 'Дата и время', 'Результат звонка',
        'Расшифровка', 'Комментарий', 'EMAIL со слов клиента',
        'Время разговора в поминутной тарификации', 'Статус звонка']

# Найти индекс колонки 'Комментарий'
idx = cols.index('Комментарий')

# Вставить в переменную с порядком колонок новую колонку после колонки 'Комментарий'
cols.insert(idx + 1, 'Расшифровка нецелевой')

# Перестроить с обновленным порядком колонок
oline_cs = oline_cs[cols]


In [17]:
# переводим колонку олайна в формат даты и времени
oline_cs['Дата и время'] = pd.to_datetime(oline_cs['Дата и время'])

In [19]:
# обозначаем переменную равную последнему дню, заргуженному у олайна
oline_last_date = last_dates.loc[last_dates['KC'] == 'oline', 'last_date'].iloc[0].date()

In [21]:
#выбираем только даты, которые больше последней даты по олайну
oline_cs = oline_cs[oline_cs['Дата и время'].dt.date > oline_last_date]

In [23]:
if oline_cs['id клиента'].count() == 0:
    print('Данные за вчерашний день отсутствуют!')
else:
    #вставляем в датафрейм с последними датами ту дату, которой ещё не было и которую мы загружаем в гугл документ
    last_dates.loc[last_dates['KC'] == 'oline', 'last_date'] = oline_cs['Дата и время'].max().strftime("%d.%m.%Y")
    print(f'{oline_cs['Ссылка на звонок'].count()} строк готовы к загрузке!')

2850 строк готовы к загрузке!


In [373]:
# убираем первую цифру от телефона
oline_cs['Номер телефона'] = oline_cs['Номер телефона'].astype(str).str[1:]

In [557]:
# определяем целевую ячейку

# Получение всех значений в нужном столбце, например, в колонке A
column_values = REPORT_CS.col_values(1)  # Нумерация столбцов с 1 (A=1)
# Поиск первой пустой строки
first_empty_row = len(column_values) + 1  # следующая строка после последней непустой
# Вставка с этой строки и колонки A (1)
start_cell_2 = f'A{first_empty_row}'

In [563]:
# загрузка в гугл

for col in oline_cs.select_dtypes(include=['datetime', 'timedelta']).columns:
    oline_cs[col] = oline_cs[col].dt.strftime("%d.%m.%Y %H:%M:%S")
        # Заменяем NaN значениями пустыми строками
oline_cs.fillna('', inplace=True)  

REPORT_CS.update(start_cell_2, oline_cs.values.tolist())

print(f'Данные загружены с {start_cell_2} ячейки {pd.to_datetime("today")}')

C:\Users\1\AppData\Local\Temp\ipykernel_15500\3406862280.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  oline_cs[col] = oline_cs[col].dt.strftime("%d.%m.%Y %H:%M:%S")
C:\Users\1\AppData\Local\Temp\ipykernel_15500\3406862280.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  oline_cs.fillna('', inplace=True)
C:\Users\1\AppData\Local\Temp\ipykernel_15500\3406862280.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#re

Данные загружены с A466 ячейки 2025-11-21 17:29:04.896420


In [10]:
# обновляем файл с последними датами по двум КЦ
last_dates.to_excel(r'C:\Users\1\Последние_даты_отчёт_КЦ.xlsx', index=False)